## INITIAL MODELS FOR Physionet

In [1]:
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from mne.decoding import CSP
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, GridSearchCV, GroupKFold
import os
import gc

In [2]:
import importlib
import load_data_physionet

importlib.reload(load_data_physionet)
from load_data_physionet import prepare_physionet_dataset

In [3]:
data_dir = "datasets/physionet/physionet.org/files/eegmmidb/1.0.0/"

In [4]:
def load_subject_data(data_dir):
    subject_dirs = sorted([d for d in os.listdir(data_dir) if d.startswith('S') and len(d) == 4])

    data_by_subject = {}

    for subj in subject_dirs:

        X_subj, y_subj = [], []

        subject_path = os.path.join(data_dir, subj)
        edf_files = sorted([f for f in os.listdir(subject_path) if f.endswith('.edf')])
        edf_files = [f for f in edf_files if not any(f.endswith(f'R0{i}.edf') for i in [1,2])]

        for edf_file in edf_files:
            edf_path = os.path.join(subject_path, edf_file)
            data = prepare_physionet_dataset(edf_path)

            try:
                data = prepare_physionet_dataset(edf_path)
                X_subj.append(data['X'])
                y_subj.append(data['y'])
            except Exception as e:
                print(f"Skipping {subj}/{edf_file}: {e}")

        if len(X_subj) == 0:
            print(f"No valid trials for {subj}, skipping subject.")
            continue

        X_subj = np.concatenate(X_subj).astype(np.float32)
        y_subj = np.concatenate(y_subj)

        data_by_subject[subj] = (X_subj, y_subj)

        gc.collect()

    return data_by_subject


In [5]:
from sklearn.base import clone

def run_loso(data_by_subject, pipe):

    subject_list = list(data_by_subject.keys())
    fold_accuracies = []

    for fold, test_subject in enumerate(subject_list):

        print(f"\nFOLD {fold} - Test: {test_subject}")

        X_train, y_train = [], []
        X_val, y_val = data_by_subject[test_subject]

        for subj in subject_list:
            if subj != test_subject:
                X_subj, y_subj = data_by_subject[subj]
                X_train.append(X_subj)
                y_train.append(y_subj)

        X_train = np.concatenate(X_train)
        y_train = np.concatenate(y_train)

        model = clone(pipe)
        model.fit(X_train, y_train)

        acc = model.score(X_val, y_val)
        fold_accuracies.append(acc)

        print(f"Val Accuracy: {acc:.4f}")

        del X_train, y_train
        gc.collect()

    mean_acc = np.mean(fold_accuracies)

    X_all = np.concatenate([data_by_subject[s][0] for s in subject_list])
    y_all = np.concatenate([data_by_subject[s][1] for s in subject_list])

    final_model = clone(pipe)
    final_model.fit(X_all, y_all)

    return final_model, mean_acc, fold_accuracies


In [6]:
transpose = FunctionTransformer(lambda X: np.transpose(X, (0, 2, 1)), validate=False)

pipe1 = Pipeline([
    ('transpose', transpose),
    ('csp', CSP(n_components=4, reg=None, log=True, norm_trace=False)),
    ('clf', LinearDiscriminantAnalysis())
])

pipe2 = Pipeline([
    ('transpose', transpose),
    ('csp', CSP(n_components=4, reg=None, log=True, norm_trace=False)),
    ('clf', SVC(kernel='rbf', C=1.0, gamma='scale'))
])

In [7]:
data_by_subject = load_subject_data(data_dir)


No valid trials for S061, skipping subject.
No valid trials for S067, skipping subject.
No valid trials for S068, skipping subject.
No valid trials for S069, skipping subject.
No valid trials for S070, skipping subject.
No valid trials for S071, skipping subject.
No valid trials for S072, skipping subject.
No valid trials for S073, skipping subject.
No valid trials for S074, skipping subject.
No valid trials for S075, skipping subject.
No valid trials for S076, skipping subject.
No valid trials for S077, skipping subject.
No valid trials for S078, skipping subject.
No valid trials for S079, skipping subject.
No valid trials for S080, skipping subject.
No valid trials for S081, skipping subject.
No valid trials for S082, skipping subject.
No valid trials for S083, skipping subject.
No valid trials for S084, skipping subject.
No valid trials for S085, skipping subject.
No valid trials for S086, skipping subject.
No valid trials for S087, skipping subject.
No valid trials for S088, skippi

/home/alumno/Desktop/datos/nn/EEGNet-Motor-Imagery-BCI/load_data_physionet.py:6: RuntimeWarning: Number of records from the header does not match the file size (perhaps the recording was not stopped before exiting). Inferring from the file size.
  raw = mne.io.read_raw_edf(edf_file, preload=True, verbose=False)
/home/alumno/Desktop/datos/nn/EEGNet-Motor-Imagery-BCI/load_data_physionet.py:6: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_edf(edf_file, preload=True, verbose=False)
/home/alumno/Desktop/datos/nn/EEGNet-Motor-Imagery-BCI/load_data_physionet.py:6: RuntimeWarning: Number of records from the header does not match the file size (perhaps the recording was not stopped before exiting). Inferring from the file size.
  raw = mne.io.read_raw_edf(edf_file, preload=True, verbose=False)
/home/alumno/Desktop/datos/nn/EEGNet-Motor-Imagery-BCI/load_data_physionet.py:6: RuntimeWarning: Limited 1 annotation(s) that were expanding o

In [8]:
model, mean_acc, fold_accs = run_loso(data_by_subject, pipe1)


FOLD 0 - Test: S001
Computing rank from data with rank=None
    Using tolerance 0.00089 (2.2e-16 eps * 10 dim * 4e+11  max singular value)
    Estimated rank (data): 10
    data: rank 10 computed from 10 data channels with 0 projectors
Reducing data rank from 10 -> 10
Estimating class=0 covariance using EMPIRICAL
Done.
Estimating class=1 covariance using EMPIRICAL
Done.
Val Accuracy: 0.5167

FOLD 1 - Test: S002
Computing rank from data with rank=None
    Using tolerance 0.0009 (2.2e-16 eps * 10 dim * 4e+11  max singular value)
    Estimated rank (data): 10
    data: rank 10 computed from 10 data channels with 0 projectors
Reducing data rank from 10 -> 10
Estimating class=0 covariance using EMPIRICAL
Done.
Estimating class=1 covariance using EMPIRICAL
Done.
Val Accuracy: 0.5357

FOLD 2 - Test: S003
Computing rank from data with rank=None
    Using tolerance 0.00089 (2.2e-16 eps * 10 dim * 4e+11  max singular value)
    Estimated rank (data): 10
    data: rank 10 computed from 10 data c

In [9]:
mean_acc

0.5191577180621522